# Knowledge graph: Neo4j and RDF

Input: `keywords_dict_expanded.pkl` from `notebooks/keyword_normalization.ipynb`.

1. Neo4j: Brand, Review, Sentiment, Aspect and Keyword nodes. Queries: `queries/cypher_queries.cypher`.
2. RDF: the same graph as Turtle (`review_kg.ttl`). Brands use Wikidata URIs, so `queries/sparql_query.rq`
   can join the local graph with company data from Wikidata.

Set `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD` as environment variables before running.

In [ ]:
import os

import pandas as pd
from neo4j import GraphDatabase

from rdflib.namespace import RDF, XSD
from rdflib import Graph, Literal, Namespace

In [ ]:
df_path = "keywords_dict_expanded.pkl"  # output of keyword_normalization.ipynb
df = pd.read_pickle(df_path)

### Neo4J

In [ ]:
# Credentials come from environment variables, never from the code
URI = os.environ["NEO4J_URI"]
USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
PASSWORD = os.environ["NEO4J_PASSWORD"]

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def create_knowledge_graph(tx, row):
    # --- get the values from the dataframe
    brand = row['BRAND']
    review_num = row['REVIEW_NUM']
    sentiment = row['SENTIMENT']
    keyword = row['KEYWORDS']
    aspect = row['ASPECT']
    subjectivity = row['SUBJECTIVITY']

    # --- run cipher queries
    tx.run("""
        // create nodes
        MERGE (b:Brand {name: $brand_name})
        
        MERGE (r:Review {reviewID: $review_id})
        ON CREATE SET r.subjectivity = $subjectivity_val        // add subjectivity as property

        MERGE (s:Sentiment {type: $sentiment_type})
                            
        MERGE (a:Aspect {name: $aspect_name})
                            
        MERGE (k:Keyword {word: $keyword_word})

        // link nodes
        MERGE (b)-[:HAS_REVIEW]->(r)
        MERGE (r)-[:EXPRESSES_SENTIMENT]->(s)
        MERGE (r)-[:RELATES_TO_ASPECT]->(a)
        MERGE (r)-[:CONTAINS_KEYWORD]->(k)
    """,
    # --- pass all the parameters
    brand_name=brand,
    review_id=review_num,
    sentiment_type=sentiment,
    aspect_name=aspect,
    keyword_word=keyword,
    subjectivity_val=subjectivity
    )


with driver.session() as session:
    # --- Iterate over DataFrame rows and create graph elements
    for index, row in df.iterrows():
        session.execute_write(create_knowledge_graph, row)

driver.close()
print("Knowledge graph creation complete!")

### RDF

In [ ]:
g = Graph()

# --- define the namespaces
EX = Namespace("http://example.org/reviews/ontology#")
DATA = Namespace("http://example.org/reviews/data/")
SCHEMA = Namespace("http://schema.org/")
WD = Namespace("http://www.wikidata.org/entity/")  # brand URIs link to Wikidata

# --- bind namespaces
g.bind("ex", EX)
g.bind("data", DATA)
g.bind("rdf", RDF)       
g.bind("xsd", XSD)       
g.bind("schema", SCHEMA) 
g.bind("wd", WD)


created_common_entities = set()
for index, row in df.iterrows():
    # Generate a unique URI for each review instance
    review_id = row["REVIEW_NUM"]
    review_uri = DATA[f"review_{review_id}"]
    # define review class: 'schema:Review'
    g.add((review_uri, RDF.type, SCHEMA.Review))

    ### Brand
    brand_name = row["BRAND"]
    if brand_name == "BK":
        brand_uri = WD.Q177054   # Burger King
        official_brand_name = "Burger King"
    else:
        brand_uri = WD.Q38076    # McDonald's
        official_brand_name = "McDonald's"
    
    if brand_uri not in created_common_entities:
        # define brand class: schema:Brand Class
        g.add((brand_uri, RDF.type, SCHEMA.Brand))
        # define official schema:name property
        g.add((brand_uri, SCHEMA.name, Literal(official_brand_name)))

        created_common_entities.add(brand_uri)
    
    # Link the review to the brand using schema:itemReviewed
    g.add((review_uri, SCHEMA.itemReviewed, brand_uri))

    ### Sentiment
    sentiment_type = str(row['SENTIMENT'])
    sentiment_uri = DATA[f"sentiment_{sentiment_type}"]
    
    if sentiment_uri not in created_common_entities:
        # define sentiment class: ex:Sentiment Class
        g.add((sentiment_uri, RDF.type, EX.Sentiment))
        # define sentiment property: property
        g.add((sentiment_uri, EX.sentimentType, Literal(sentiment_type)))
        created_common_entities.add(sentiment_uri)
    
    # Link the review to the sentiment using ex:expressesSentiment
    g.add((review_uri, EX.expressesSentiment, sentiment_uri))

    ### Aspect
    aspect_name = str(row['ASPECT'])
    aspect_uri = DATA[f"aspect_{aspect_name}"]
    
    if aspect_uri not in created_common_entities:
        # define aspect class: ex:Aspect Class
        g.add((aspect_uri, RDF.type, EX.Aspect))
        # define aspect property: property
        g.add((aspect_uri, EX.aspectName, Literal(aspect_name)))
        created_common_entities.add(aspect_uri)
    
    # Link the review to the aspect using ex:relatesToAspect
    g.add((review_uri, EX.relatesToAspect, aspect_uri))

    ### Keyword
    keyword = row['KEYWORDS']
    keyword_word = str(keyword)
    keyword_uri = DATA[f"keyword_{keyword_word}"]
    
    if keyword_uri not in created_common_entities:
        # define keyword class: ex:Aspect Class
        g.add((keyword_uri, RDF.type, EX.Keyword))
        # define keyword property: ex:keywordValue property
        g.add((keyword_uri, EX.keywordValue, Literal(keyword_word)))
        created_common_entities.add(keyword_uri)
    
    # Link the review to the keyword using a custom ex:mentionsKeyword Object Property
    g.add((review_uri, EX.containsKeyword, keyword_uri))

    ### Subjectivity
    subjectivity_value = row['SUBJECTIVITY']
    # added as a property to the review instance
    g.add((review_uri, EX.withSubjectivity, Literal(float(subjectivity_value), datatype=XSD.float)))


print(f"RDF graph built: {len(g)} triples")


In [ ]:
g.serialize(destination="review_kg.ttl", format="turtle")
print("\nRDF graph saved to review_kg.ttl")